# 4 — policy RAG

The agent can search hotels and price a stay, but it knows nothing about the
rules around them — cancellation fees, pet weight limits, check-in hours.
Ask it one of those questions now and it will answer anyway, because nothing
stops it from guessing. This book gives it a rulebook it can actually consult,
without stuffing the whole thing into every prompt.

In [1]:
import os

from hotelbot.config import configure_tracing
from hotelbot.agent import build_agent, run

configure_tracing(book=4)
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY is needed later in this book, for embeddings"

agent = build_agent()

**The model guesses policy.** Ask it something only `data/policies/pets.md`
actually answers: "Can I bring my dog to Casa do Castelo, and what would it
cost?" Predict first — refuse outright, hedge with "check with the hotel", or
invent a specific fee? Then run it and open `data/policies/pets.md` and
compare by eye. The document says a flat €25 per stay and a 20 kg limit;
the model has no way to know either number.

In [2]:
outcome = run(agent, "Can I bring my dog to Casa do Castelo, and what would it cost?")
print(outcome.structured_response)

text='No — Casa do Castelo doesn\'t have the "pets" amenity, so dogs aren\'t allowed there. If you\'d like, Principe Real Boutique (€178/night) or Graca Residence (€200/night), both in Lisbon, do allow pets for a flat €25/stay fee (pets up to 20kg, with leash/crate rules in shared spaces).'


**Documents become chunks.** A file like `pets.md` answers several
different questions in different paragraphs; handing the model the whole
file for a one-line question like "can I bring a dog" buries the fee and the
weight limit in three paragraphs it didn't ask about. `RecursiveCharacterTextSplitter`
cuts each file into overlapping pieces instead — `chunk_size=500` characters
per piece, `chunk_overlap=50` so a sentence split across a chunk boundary
still appears whole in at least one of the two chunks. Load the five files,
split them, and print the chunk count, then two adjacent chunks from the
same file so the overlap is visible.

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

from hotelbot.config import CHUNK_OVERLAP, CHUNK_SIZE, POLICIES_DIR

splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

chunks = []
for path in sorted(POLICIES_DIR.glob("*.md")):
    for text in splitter.split_text(path.read_text()):
        chunks.append({"source": path.name, "text": text})

print(len(chunks), "chunks from", len(list(POLICIES_DIR.glob('*.md'))), "files")
print()
print(chunks[0]["text"])
print("---")
print(chunks[1]["text"])

11 chunks from 5 files

# Accessibility policy

Roughly one in five rooms across our properties is step-free from the
street to the room door, including step-free bathrooms with grab rails and
a roll-in shower. These rooms are not shown separately in search yet, so a
guest who needs one should be flagged for a manual check against the
hotel's own accessible-room count.
---
Properties with an "elevator" amenity guarantee lift access to every floor
they operate; properties without it are walk-up only above the ground
floor, with no exceptions arranged on the day.

Guide and assistance dogs are permitted everywhere pets otherwise are not,
including the pool area and gym, and are exempt from the pet fee described
in the pet policy.


**An embedding is a list of numbers.** `OpenAIEmbeddings().embed_query("dog")`
turns the word into 1536 floats — not a lookup, a position. Print the first
five so "a list of numbers" stops being abstract.

In [4]:
from langchain_openai import OpenAIEmbeddings

from hotelbot.config import EMBEDDING_MODEL

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
dog_vector = embeddings.embed_query("dog")

print("length:", len(dog_vector))
print("first five:", dog_vector[:5])

length: 1536
first five: [0.051177978515625, -0.018707275390625, -0.0043487548828125, 0.07269287109375, -0.00711822509765625]


**Near in meaning, near in numbers.** Embed "pet" and "cancellation" too.
A dot product multiplies two vectors position by position and adds the
results — `sum(a[i] * b[i] for i in range(len(a)))` — one number that's
larger when two vectors point the same way. Compute it by hand with a `for`
loop for ("dog", "pet") and ("dog", "cancellation"), then divide each by the
two vectors' lengths (`sqrt(sum(x*x for x in v))`) to normalise — that turns
the raw dot product into cosine similarity, which stays between -1 and 1
regardless of how long the vectors happen to be. Predict which pair scores
higher before running.

In [5]:
import math


def dot(a, b):
    return sum(x * y for x, y in zip(a, b))


def norm(v):
    return math.sqrt(dot(v, v))


def cosine_similarity(a, b):
    return dot(a, b) / (norm(a) * norm(b))


pet_vector = embeddings.embed_query("pet")
cancellation_vector = embeddings.embed_query("cancellation")

print("dog . pet          (raw dot):", dot(dog_vector, pet_vector))
print("dog . cancellation  (raw dot):", dot(dog_vector, cancellation_vector))
print()
print("dog . pet          (cosine):", cosine_similarity(dog_vector, pet_vector))
print("dog . cancellation  (cosine):", cosine_similarity(dog_vector, cancellation_vector))

dog . pet          (raw dot): 0.6385277707503292
dog . cancellation  (raw dot): 0.13162363107676356

dog . pet          (cosine): 0.6386568422934442
dog . cancellation  (cosine): 0.1315956633636645


**The index.** A vector store is what makes "nearest chunks" a search
instead of a manual scan: `Chroma` holds every chunk's embedding and, given a
new piece of text, returns the chunks whose vectors are closest to it.
`Chroma(collection_name, embedding_function)` opens a collection;
`add_documents` embeds each chunk and stores it. With no `persist_directory`
this collection lives only in memory and disappears when the kernel stops.
`reset_collection()` first makes this cell safe to rerun — without it,
running the cell twice would store every chunk twice.

Predict which policy file's chunks answer "bringing a dog" before running,
then call `similarity_search("bringing a dog", k=3)` and print the three
chunks' source files and first lines.

In [ ]:
import time

from langchain_chroma import Chroma
from langchain_core.documents import Document

documents = [Document(page_content=c["text"], metadata={"source": c["source"]}) for c in chunks]

start = time.time()
index = Chroma(collection_name="policies_demo", embedding_function=embeddings)
index.reset_collection()
index.add_documents(documents)
build_seconds = time.time() - start
print(f"built in {build_seconds:.2f}s")

print()
for doc in index.similarity_search("bringing a dog", k=3):
    print(doc.metadata["source"], "-", doc.page_content.splitlines()[0])

**Persistence.** That build paid for embedding every chunk, and it pays again
every time the cell runs. `hotelbot.index.get_policy_index()` builds the same
index into `data/chroma/` on disk instead, and the next call just reopens it,
unless the policy files have changed since.

To notice a change, each file gets a fingerprint: a sha256 hash, which turns
the file's contents into a 64-character string. The same contents always
give the same string, and changing a single character gives a completely
different one. Every stored chunk keeps the fingerprint of the file it came
from. `get_policy_index()` compares those with the fingerprints of the files
on disk now, and rebuilds if a file was added, removed, or edited.

Print the fingerprints, then call `get_policy_index()` twice and time both.
Predict before running: on a first run, with no `data/chroma/` yet, which
call is slow and which is fast? What changes if you run the cell again?

In [ ]:
import os

from hotelbot.config import CHROMA_DIR
from hotelbot.index import get_policy_index, policy_fingerprints

for name, digest in policy_fingerprints().items():
    print(f"{name:<18} {digest[:16]}...")
print()

start = time.time()
get_policy_index()
first_seconds = time.time() - start

start = time.time()
reopened = get_policy_index()
second_seconds = time.time() - start

print(f"first call:  {first_seconds:.3f}s")
print(f"second call: {second_seconds:.3f}s")
print()
print(sorted(os.listdir(CHROMA_DIR)))

**The tool.** `hotelbot.tools.lookup_policy` wraps the index as a tool: it
takes a question, runs `similarity_search(question, k=3)` on the index from
`get_policy_index()`, and joins the three chunks into one string, each
starting with `[source: <file>]`. Call it directly, the way the agent will.
The string it prints is all the model ever sees of the index. The model never
sees vectors, scores or the Chroma collection, just this text, returned as a
tool result like `search_hotels`'s list of hotels.

In [8]:
from hotelbot.tools import lookup_policy

print(lookup_policy.invoke({"question": "can I bring a dog"}))

[source: pets.md]
# Pet policy

Dogs and cats are welcome at properties tagged with the "pets" amenity — check
a hotel's amenity list before promising a guest they can bring one along.
Other animals are handled case by case; ask the front desk directly.

There is a flat fee of €25 per stay, per pet, regardless of how many nights
the stay covers. This is charged at check-in, not folded into the room rate
shown during search.

[source: pets.md]
Pets are limited to 20 kg. A heavier animal needs the property's written
sign-off in advance, arranged directly with the hotel rather than through
booking.

Pets must be leashed or crated in all shared indoor spaces (lobby,
breakfast room, corridors) and are not permitted in the pool area or gym
under any circumstances. A pet left alone in a room must be crated, and
housekeeping will skip a room where an uncrated pet is present alone.

[source: accessibility.md]
Properties with an "elevator" amenity guarantee lift access to every floor
they operat

**The agent decides to look it up.** `build_agent()` now passes
`lookup_policy` to the model alongside the catalogue tools. Nothing tells the
model to call it on any particular question. The tool's docstring says it
covers rules, fees, limits and hours, and the model reads that the same way
it read `search_hotels`'s. Cell 2 showed Casa do Castelo doesn't take pets at
all, so ask about Principe Real Boutique, which does. Then the fee and weight
limit actually matter.

`tools_called` lists the name of every tool call in the run, in order. The
last one is the typed answer from book 3 (`PlainAnswer` or
`BookingProposal`). The ones before it are the lookups. Check the answer
against `pets.md`: €25 per stay, 20 kg.

In [9]:
def tools_called(outcome):
    return [call["name"] for m in outcome.messages for call in getattr(m, "tool_calls", [])]


agent = build_agent()

outcome = run(agent, "Can I bring my dog to Principe Real Boutique, and what would it cost?")
print(tools_called(outcome))
print()
print(outcome.structured_response)

['search_hotels', 'lookup_policy', 'PlainAnswer']

text="Yes — Principe Real Boutique is pet-friendly (up to 20 kg; heavier dogs need prior written sign-off from the hotel). There's a flat €25 per-stay pet fee, charged at check-in on top of the room rate (€178/night). Want me to check dates for you?"


**The agent decides not to.** "Is there a gym at Casa do Castelo?" is a
question about one hotel's amenities, and `get_hotel` / `search_hotels`
already return amenity lists. Predict before running: does
`lookup_policy` appear in the list of tools called?

In [10]:
outcome = run(agent, "Is there a gym at Casa do Castelo?")
print(tools_called(outcome))
print()
print(outcome.structured_response)

['search_hotels', 'PlainAnswer']

text='Yes — Casa do Castelo (Alfama, Lisbon) has a gym, along with breakfast and wifi.'


**A question on the boundary.** "Do I pay now or at the hotel?" doesn't name
a policy, and it sounds like general knowledge that most hotels share. The
real answer is in `payment.md`: a 20% deposit at booking, the other 80% at
check-in. Predict whether the agent looks it up or answers from what it
assumes hotels usually do. Whichever it does, the list of tools called shows
which one happened. If it guessed, compare its answer with `payment.md`. That
mismatch is the same gap as cell 2, and it is still there whenever the model
doesn't think to look.

In [11]:
outcome = run(agent, "Do I pay now or at the hotel?")
print(tools_called(outcome))
print()
print(outcome.structured_response)

['lookup_policy', 'PlainAnswer']

text='Both: a 20% deposit is charged now to confirm the booking, and the remaining 80% is paid at the hotel at check-in (card or cash, local currency). A local tourist tax (€2/person/night) is also added at check-in.'


**What you can do now.** The agent can answer from five policy documents
instead of guessing, and it picks which questions need them. Building the
index costs embedding calls once. After that, `get_policy_index()` opens the
copy saved in `data/chroma/`.

What the agent reads comes back as an ordinary `ToolMessage`, the same
channel as a hotel search result, and the model trusts it the same way. If
someone edits a policy document to contain instructions, the model reads
those too. Book 7 shows why that channel needs watching. Book 5 comes first:
the agent so far forgets each conversation as soon as it ends and can only
propose a booking, so book 5 gives it memory across turns and lets it
actually book.